# 检查分钟行情缺口

## 目标

得到一张有明确区间边界、实际价格和缺失标记的表。这里只审查时间覆盖，不计算波动率或补造行情。

本文件使用虚构教学数据，不是论文复现或生产数据。

## 准备

使用 Python 3.10+ 内核，按顺序运行全部单元格。计算仅依赖标准库，无需密钥、联网或额外数据文件。可在已有的 Jupyter 环境中打开。

输入已内嵌，与同目录 inputs.json 内容一致；可在下一个单元格中修改 args 试验。时间与单位必须显式保留。

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"minute-bar-gaps\",\"identity\":\"synthetic\",\"args\":[[{\"openTime\":\"2025-01-06T01:30:00Z\",\"close\":100},{\"openTime\":\"2025-01-06T01:40:00Z\",\"close\":102}],[\"2025-01-06T01:30:00Z\",\"2025-01-06T01:35:00Z\",\"2025-01-06T01:40:00Z\"],5],\"expected\":[{\"openTime\":\"2025-01-06T01:30:00.000Z\",\"endExclusive\":\"2025-01-06T01:35:00.000Z\",\"close\":100,\"status\":\"observed\"},{\"openTime\":\"2025-01-06T01:35:00.000Z\",\"endExclusive\":\"2025-01-06T01:40:00.000Z\",\"close\":null,\"status\":\"missing\"},{\"openTime\":\"2025-01-06T01:40:00.000Z\",\"endExclusive\":\"2025-01-06T01:45:00.000Z\",\"close\":102,\"status\":\"observed\"}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## 步骤

### 1. 冻结时间含义

先核对供应商的时间戳表示开盘还是收盘，再转成示例要求的UTC整秒格式。示例把区间写成左闭右开，endExclusive不是供应商原始close_time的替代字段。真实分钟数据的原始时区、边界含义与字段应另行保留，不能只改列名就当作完成映射。

### 2. 独立准备预期网格

根据交易所日历和会话规则提供expectedOpens，只放已经结束且应当观察的区间。午休、假期或闭市不应该算缺口；停牌需要额外状态，交易日历本身不够。不要从已有行情反推应有网格，否则缺失的区间会在检查之前就消失。

### 3. 精确连接并拒绝冲突

按区间起点连接，重复时间、重叠网格、网格外观测和无效价格直接拒绝。预期网格可以有休市间隔，但单个区间不能重叠。函数不猜测最近一分钟，也不把两个不同证券的同一时间合并，因此必须事先按证券和场所分批。

### 4. 保留缺失，再解释原因

虚构网格有三个5分钟区间，中间一条没有观测，输出missing和null。这里不能判断是停牌、无交易还是采集失败；应结合原始回执与市场状态另查。后续计算收益前，必须选择并说明跨缺口处理，不能把null换成0或上一条价格后隐瞒处理。

### 方法与假设

- 不检查尚未结束的K线；缺少成交不自动等于采集失败。
- 日历只描述市场开放，不能证明证券没有停牌。
- 样本不执行插值、补零或向前填充。

In [ ]:
from datetime import datetime, timedelta, timezone
import math


def audit_bar_grid(rows, expected_opens, interval_minutes):
    def parse(value):
        try:
            parsed = datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
            if parsed.strftime("%Y-%m-%dT%H:%M:%SZ") != value:
                raise ValueError()
            return parsed
        except (ValueError, TypeError):
            raise ValueError("utc_seconds_required")

    if isinstance(interval_minutes, bool) or not isinstance(interval_minutes, (int, float)) or not math.isfinite(interval_minutes) or int(interval_minutes) != interval_minutes or not 0 < interval_minutes <= 1440:
        raise ValueError("invalid_interval")
    if not expected_opens or len(expected_opens) > 10000:
        raise ValueError("invalid_grid")
    grid = sorted(map(parse, expected_opens))
    interval = timedelta(minutes=interval_minutes)
    if any(current - previous < interval for previous, current in zip(grid, grid[1:])):
        raise ValueError("overlapping_grid")
    slots, observations = set(grid), {}
    for row in rows:
        time = parse(row.get("openTime"))
        if time not in slots:
            raise ValueError("outside_grid")
        if time in observations:
            raise ValueError("duplicate_bar")
        close = row.get("close")
        if isinstance(close, bool) or not isinstance(close, (int, float)) or not math.isfinite(close) or close <= 0:
            raise ValueError("invalid_close")
        observations[time] = close
    iso = lambda value: value.isoformat(timespec="milliseconds").replace("+00:00", "Z")
    return [{"openTime": iso(time), "endExclusive": iso(time + interval), "close": observations.get(time), "status": "observed" if time in observations else "missing"} for time in grid]


### 运行小样本

输出3行：observed、missing、observed；收盘价100、null、102。原始两行不被修改。

In [ ]:
result = audit_bar_grid(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 检查

将每一行与网页示例的预期输出比较。修改输入后，断言失败可能正是预期结果：先解释差异，不要直接删除验证。

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("通过：结果与网页虚构示例一致。")

## 下一步

真实数据须先通过已认证 GET /v1/catalog 核对权限、字段、schema_major、窗口与来源，再按实际合同映射。这里列出的是候选输入身份，不保证可用或历史完整。不要把 API as_of 当作历史财报版本。真实输入替换后须重新验证；不要沿用这份小样本的通过结论。

- `cn.dataset.stk_mins`
- `cn.market.trade_calendar`

### 参考资料

- [Tushare：股票历史分钟](https://tushare.pro/document/2?doc_id=370)
- [Andersen等：已实现波动的构造](https://users.ssc.wisc.edu/~behansen/718/Anderson2003.pdf)

[返回教程](https://tradingdatas.com/recipes/minute-bar-gaps/)